# Get a Corpus Overview

Understand what's in your video collection -- themes, subjects, patterns, and key statistics.
Use this when starting work with a new collection, feeding context to a downstream agent, or generating collection summaries for a UI.

In [ ]:
import json
import os

import requests

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "YOUR_API_KEY")
BASE_URL = "https://api.twelvelabs.io/v1.3"
HEADERS = {"x-api-key": API_KEY, "Content-Type": "application/json"}

# Replace with your knowledge store ID
STORE_ID = "your_knowledge_store_id"

## Helper Functions

A utility to extract text content from a Jockey API response.

In [ ]:
def parse_response(result: dict) -> str | dict:
    """Extract text content from a Jockey API response."""
    for output in result["output"]:
        if output["type"] == "message":
            for content in output["content"]:
                return content["text"]
    return ""

## Plain Text Overview

The simplest approach: ask Jockey for a free-form summary of your video collection.
This returns a natural-language overview covering main themes, recurring subjects, content types, and notable patterns.

In [ ]:
response = requests.post(
    f"{BASE_URL}/responses",
    headers=HEADERS,
    json={
        "model": "jockey1.0",
        "input": [
            {
                "type": "message",
                "role": "user",
                "content": (
                    "Give me a comprehensive overview of this video collection. "
                    "Include main themes, recurring subjects, content types, "
                    "and any notable patterns."
                ),
            }
        ],
        "knowledge_store_id": STORE_ID,
    },
)

result = response.json()
print(parse_response(result))

## Structured Overview Schema

For programmatic consumption, define a JSON schema that tells Jockey exactly what fields to return.
This schema captures the total video count, dominant themes, content types, key subjects, notable patterns, and a narrative summary.

In [ ]:
OVERVIEW_SCHEMA = {
    "type": "object",
    "properties": {
        "total_videos": {"type": "integer"},
        "themes": {"type": "array", "items": {"type": "string"}},
        "content_types": {"type": "array", "items": {"type": "string"}},
        "key_subjects": {"type": "array", "items": {"type": "string"}},
        "patterns": {"type": "array", "items": {"type": "string"}},
        "summary": {"type": "string"},
    },
}

## Structured Overview Request

Pass the schema via the `text` parameter to receive a structured JSON response.
The response text is valid JSON that can be parsed and used directly in downstream pipelines.

In [ ]:
response = requests.post(
    f"{BASE_URL}/responses",
    headers=HEADERS,
    json={
        "model": "jockey1.0",
        "input": [
            {
                "type": "message",
                "role": "user",
                "content": "Give me a structured overview of this video collection.",
            }
        ],
        "knowledge_store_id": STORE_ID,
        "text": {"format": {"type": "json_schema", "name": "corpus_overview", "schema": OVERVIEW_SCHEMA}},
    },
)

result = response.json()
overview = json.loads(parse_response(result))

print(f"Videos: {overview['total_videos']}")
print(f"Themes: {', '.join(overview['themes'])}")
print(f"Content Types: {', '.join(overview['content_types'])}")
print(f"Key Subjects: {', '.join(overview['key_subjects'])}")
print(f"Patterns: {', '.join(overview['patterns'])}")
print(f"\nSummary: {overview['summary']}")

## Next Steps

- **[Search Videos](search_videos.ipynb)** -- find specific moments in your collection
- **[Extract Entities](extract_entities.ipynb)** -- list all people, places, objects, and concepts
- **[Find Organization Axes](find_organization_axes.ipynb)** -- discover the best ways to categorize your videos
- **[Enrich Content](enrich_content.ipynb)** -- get richer, domain-specific metadata

See also:
- [Structured Output Guide](../../docs/guides/structured-output.md) -- more on JSON schema responses